# 01. Ingestão e Organização dos Dados Brutos

**Projeto:** Sanidade Vegetal (SugarVision) — Detecção de Doenças em Cana-de-Açúcar  
**Disciplina:** MAQ-027 (Aprendizado de Máquina II)  
**Fase SEMMA:** Sample & Explore | **Sprint 1**  

---

### 🎯 Objetivos deste Notebook:
1. **Mapeamento e Ingestão:** Varrer os diretórios em `data/raw/` catalogando todos os arquivos de imagem das fontes (*Roboflow* e *Mendeley Data*).
2. **Validação de Integridade:** Testar a leitura de cada arquivo com `PIL.Image` / `OpenCV`, detectando imagens corrompidas ou com extensões inválidas.
3. **Metadados dos Dados Brutos:** Extrair resolução (largura, altura, canais), tamanho em KB e rótulos/classes a partir da estrutura de pastas.
4. **Inspeção Visual Inicial:** Plotar uma amostragem visual estratificada por classe para conferência de sanidade vegetal.

In [3]:
# 1. Configuração e Importação de Bibliotecas
import os
from pathlib import Path
import pandas as pd
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns

# Configurações de exibição e plotagem
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
pd.set_option('display.max_colwidth', None)

# Definição dos caminhos de diretório
BASE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DATA_DIR = BASE_DIR / 'datasets'
PROCESSED_DIR = BASE_DIR / 'data' / 'processed'

print(f"[✓] Diretório Base do Projeto: {BASE_DIR}")
print(f"[✓] Diretório de Dados Brutos: {RAW_DATA_DIR}")
print(f"[✓] Diretório existe? {RAW_DATA_DIR.exists()}")

[✓] Diretório Base do Projeto: c:\Users\ELISAALMEIDAALCANTAR\Documents\Sanidade-Vegetal
[✓] Diretório de Dados Brutos: c:\Users\ELISAALMEIDAALCANTAR\Documents\Sanidade-Vegetal\datasets
[✓] Diretório existe? True


--- 
## 2. Varredura e Validação de Leitura dos Arquivos
Vamos iterar sobre todos os arquivos dentro de `data/raw/`, validar a leitura de cada imagem, coletar metadados estruturados e filtrar eventuais arquivos corrompidos.

In [ ]:
VALID_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
records = []
corrupted_files = []

def parse_class_label(file_path, dataset_source):
    """
    Normaliza a classe inspecionando o nome do arquivo (Roboflow)
    e os diretórios pais (Mendeley Data), corrigindo grifos como 'MOASIC'.
    """
    filename_lower = file_path.name.lower()
    
    # 1. Padrão Roboflow (o nome do arquivo identifica a patologia)
    if 'healthy' in filename_lower:
        return 'HEALTHY'
    elif 'mosaic' in filename_lower or 'moasic' in filename_lower:
        return 'MOSAIC'
    elif 'redrot' in filename_lower or 'red_rot' in filename_lower or 'red rot' in filename_lower:
        return 'RED ROT'
    elif 'rust' in filename_lower:
        return 'RUST'
    elif 'yellow' in filename_lower:
        return 'YELLOW LEAF'
    elif 'scald' in filename_lower:
        return 'LEAF SCALD'
    elif 'grassyshot' in filename_lower or 'grassy_shoot' in filename_lower or 'grassy' in filename_lower:
        return 'GRASSY SHOOT'
        
    # 2. Padrão Mendeley Data (a pasta identifica a classe + correção de typos)
    for part in reversed(file_path.parts[:-1]):
        part_upper = part.upper()
        if 'MOASIC' in part_upper or 'MOSAIC' in part_upper:
            return 'MOSAIC'
        elif 'HEALTHY' in part_upper:
            return 'HEALTHY'
        elif 'RED ROT' in part_upper or 'REDROT' in part_upper:
            return 'RED ROT'
        elif 'RUST' in part_upper:
            return 'RUST'
        elif 'YELLOW' in part_upper:
            return 'YELLOW LEAF'
        elif 'SCALD' in part_upper:
            return 'LEAF SCALD'
        elif 'GRASSY' in part_upper:
            return 'GRASSY SHOOT'
            
    return 'UNLABELED'

print("Iniciando validação e leitura dos arquivos...")

for file_path in RAW_DATA_DIR.rglob('*'):
    if file_path.is_file() and file_path.suffix.lower() in VALID_EXTENSIONS:
        try:
            # 1. Validação física de integridade da imagem
            with Image.open(file_path) as img:
                img.verify()
            
            # 2. Reabertura para extração de propriedades dimensionais
            with Image.open(file_path) as img:
                width, height = img.size
                mode = img.mode
                channels = len(img.getbands())
            
            relative_parts = file_path.relative_to(RAW_DATA_DIR).parts
            dataset_source = relative_parts[0] if len(relative_parts) > 1 else 'datasets_root'
            
            # Extrai a classe real padronizada
            class_label = parse_class_label(file_path, dataset_source)
            
            # Mantém registro da partição original do Roboflow (train / valid / test)
            split_partition = relative_parts[1] if dataset_source == 'roboflow_sugarcane' and len(relative_parts) > 2 else 'full'
            
            records.append({
                'filepath': str(file_path.resolve()),
                'filename': file_path.name,
                'relative_path': str(file_path.relative_to(RAW_DATA_DIR)),
                'dataset_source': dataset_source,
                'split_partition': split_partition,
                'class_label': class_label,
                'extension': file_path.suffix.lower(),
                'width': width,
                'height': height,
                'channels': channels,
                'color_mode': mode,
                'size_kb': round(file_path.stat().st_size / 1024, 2)
            })
        except Exception as e:
            corrupted_files.append((str(file_path), str(e)))

df_raw = pd.DataFrame(records)

print(f"\n[✓] Total de imagens íntegras catalogadas: {len(df_raw)}")
print(f"[!] Total de arquivos corrompidos/ilegíveis: {len(corrupted_files)}")
if corrupted_files:
    print("Primeiras falhas encontradas:", corrupted_files[:5])

Iniciando validação e leitura dos arquivos...


NameError: name 'RAW_DATA_DIR' is not defined

--- 
## 3. Registro do Formato, Dimensões e Distribuição das Classes
Exibição dos metadados estatísticos e distribuição por fonte de dados e classes patológicas.

In [5]:
if not df_raw.empty:
    print("--- Distribuição por Fonte do Dataset ---")
    print(df_raw['dataset_source'].value_counts())
    
    print("\n--- Distribuição por Classe Fitossanitária ---")
    print(df_raw['class_label'].value_counts())
    
    print("\n--- Resumo Estatístico das Dimensões ---")
    print(df_raw[['width', 'height', 'channels', 'size_kb']].describe())
else:
    print("Nenhum arquivo de imagem encontrado em data/raw/. Verifique se o download e a extração foram concluídos.")

--- Distribuição por Fonte do Dataset ---
dataset_source
roboflow_sugarcane    5551
mendeley_data         1020
Name: count, dtype: int64

--- Distribuição por Classe Fitossanitária ---
class_label
train                  4860
valid                   464
LEAF SCALD DISEASES     439
MOASIC                  375
test                    227
GRASSYSHOT              206
Name: count, dtype: int64

--- Resumo Estatístico das Dimensões ---
             width       height  channels       size_kb
count  6571.000000  6571.000000    6571.0   6571.000000
mean   1240.546949  1046.936235       3.0    843.958288
std    1645.310929  1046.660711       0.0   2287.298057
min     640.000000   640.000000       3.0     17.300000
25%     640.000000   640.000000       3.0     44.400000
50%     640.000000   640.000000       3.0     60.610000
75%     640.000000   640.000000       3.0     81.485000
max    6016.000000  6005.000000       3.0  14517.640000


--- 
## 4. Inspeção Visual Inicial (Geração da Primeira Amostra)
Plotagem de uma grade com amostras aleatórias de cada patologia foliar para validação qualitativa das imagens.

In [ ]:
def plot_class_samples(df, samples_per_class=3):
    if df.empty:
        print("DataFrame vazio. Impossível gerar inspeção visual.")
        return
    
    classes = df['class_label'].unique()
    n_classes = len(classes)
    
    fig, axes = plt.subplots(n_classes, samples_per_class, figsize=(samples_per_class * 4, n_classes * 3.5))
    if n_classes == 1:
        axes = np.expand_dims(axes, axis=0)
    
    for row_idx, cls in enumerate(classes):
        subset = df[df['class_label'] == cls]
        sample_n = min(samples_per_class, len(subset))
        sample_rows = subset.sample(sample_n, random_state=42)
        
        for col_idx in range(samples_per_class):
            ax = axes[row_idx, col_idx]
            if col_idx < len(sample_rows):
                row = sample_rows.iloc[col_idx]
                img = Image.open(row['filepath'])
                ax.imshow(img)
                ax.set_title(f"{cls}\n{row['width']}x{row['height']} | {row['size_kb']} KB", fontsize=9)
            ax.axis('off')
            
    plt.suptitle("Inspeção Visual Inicial - Amostras por Classe Fitossanitária", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

if not df_raw.empty:
    plot_class_samples(df_raw, samples_per_class=3)

--- 
## 5. Salvamento do Catálogo de Metadados Brutos
Exportamos a tabela de metadados para servir de base para as etapas de exploração estatística e preparação (SEMMA: Explore & Modify).

In [ ]:
if not df_raw.empty:
    output_dir = BASE_DIR / 'data' / 'processed'
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / 'metadata_raw_images.csv'
    
    df_raw.to_csv(output_path, index=False)
    print(f"[✓] Catálogo de metadados salvo com sucesso em: {output_path}")